Firstly, we install the rquired libraries.

In [ ]:
!pip install transformers pandas torch

now, we create our incident dataset.

In [ ]:
import pandas as pd

data = [
    "Accident on highway near junction 5, two lanes blocked, heavy congestion",
    "Road construction causing slow traffic in city center",
    "Traffic signal not working at main intersection",
    "Minor accident blocking one lane, moderate delay",
    "Heavy rain causing slow movement across all lanes"
]

df = pd.DataFrame(data, columns=["incident_text"])
df

now that we have our dataset, we run the lare langauge model Flan-t5

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

Now, we define the incident analysis function.

In [ ]:
def analyze_incident(text):

    prompt = f"""
Classify the type of traffic incident.

Choose ONLY one:
accident, roadwork, signal_failure, weather, event, other

Incident: {text}
"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    outputs = model.generate(**inputs, max_new_tokens=10)

    response = tokenizer.decode(outputs[0], skip_special_tokens=True).strip().lower()

    return response

Now, Let's build RULE-BASED EXTRACTION

In [ ]:
def extract_features(text):
    text = text.lower()

    # incident type from LLM
    incident_type = analyze_incident(text)

    # severity
    if "heavy" in text or "severe" in text:
        severity = "high"
    elif "moderate" in text:
        severity = "medium"
    else:
        severity = "low"

    # lanes blocked
    if "two" in text or "2" in text:
        lanes = 2
    elif "one" in text or "1" in text:
        lanes = 1
    else:
        lanes = 0

    # impact
    if "congestion" in text:
        impact = "congestion"
    elif "slow" in text:
        impact = "delay"
    else:
        impact = "low impact"

    return {
        "incident_type": incident_type,
        "severity": severity,
        "lanes_blocked": lanes,
        "expected_impact": impact
    }

Let's apply to the dataset.

In [ ]:
df["structured"] = df["incident_text"].apply(extract_features)

structured_df = pd.json_normalize(df["structured"])
final_df = pd.concat([df, structured_df], axis=1)

final_df.head()

Now, it's time to build the decision logic.

In [ ]:
def decision_logic(row):

    if row["incident_type"] == "accident":
        if row["severity"] == "high":
            return "Reroute traffic + emergency response"
        else:
            return "Adjust signals"

    elif row["incident_type"] == "roadwork":
        return "Pre-planned diversion"

    elif row["incident_type"] == "signal_failure":
        return "Manual control"

    else:
        return "Monitor traffic"

In [ ]:
final_df["decision"] = final_df.apply(decision_logic, axis=1)

In [ ]:
final_df[["incident_text", "incident_type", "severity", "decision"]]

finally, we add the confidence scores to the model.

In [ ]:
def confidence_score(row):
    score = 0.5

    if row["incident_type"] in ["accident", "roadwork"]:
        score += 0.2

    if row["severity"] == "high":
        score += 0.2

    if row["lanes_blocked"] > 0:
        score += 0.1

    return round(min(score, 1.0), 2)

In [ ]:
final_df["confidence"] = final_df.apply(confidence_score, axis=1)

Let's verify our system is working properly.

In [ ]:
final_df[["incident_text", "incident_type", "severity", "lanes_blocked", "decision"]]

In [ ]:
test_cases = [
    "Massive accident blocking three lanes with heavy congestion",
    "Traffic signal failure causing delays",
    "Roadwork causing moderate traffic slowdown"
]

test_df = pd.DataFrame(test_cases, columns=["incident_text"])

test_df["structured"] = test_df["incident_text"].apply(extract_features)

test_structured = pd.json_normalize(test_df["structured"])
test_final = pd.concat([test_df, test_structured], axis=1)

test_final["decision"] = test_final.apply(decision_logic, axis=1)

test_final